# Pun prolaz kroz main.ipynb

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NEMA GPU")

GPU: Tesla T4


In [ ]:
REPO = "https://github.com/AnjaCvetkovic25/oxford_flowers102.git"

!rm -rf /content/oxford_flowers102
!git clone -q $REPO /content/oxford_flowers102
%cd /content/oxford_flowers102
!git log --oneline -3
print()
!ls -la assets/ | tail -12

/content/oxford_flowers102
10780fe (HEAD -> main, origin/main, origin/HEAD) dodato jos kesiranja
6bb7232 male izmene
3855b3f dodata diskusija

-rw-r--r-- 1 root root       84 Aug 24 14:48 best_cnn_config.json
-rw-r--r-- 1 root root   565125 Aug 24 14:48 best_cnn_metrics.json
-rw-r--r-- 1 root root   708663 Aug 24 14:48 best_cnn_metrics_v2.json
-rw-r--r-- 1 root root 35338162 Aug 24 14:48 best_cnn_model.pt
-rw-r--r-- 1 root root 60161471 Aug 24 14:48 best_cnn_model_v2.pt
-rw-r--r-- 1 root root    33272 Aug 24 14:48 error_acc_vs_count.png
-rw-r--r-- 1 root root    24803 Aug 24 14:48 error_confidence.png
-rw-r--r-- 1 root root   562533 Aug 24 14:48 error_examples.png
-rw-r--r-- 1 root root   114675 Aug 24 14:48 gradcam_v2.png
-rw-r--r-- 1 root root 44995979 Aug 24 14:48 resnet18_finetuned.pt
-rw-r--r-- 1 root root 44995339 Aug 24 14:48 resnet18_head.pt
-rw-r--r-- 1 root root   116505 Aug 24 14:48 resnet18_metrics.json


In [ ]:
import os
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/102flowers.tgz"):
    !cd data && curl -sL -O https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz
if not os.path.exists("data/imagelabels.mat"):
    !cd data && curl -sL -O https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat
!ls -la data/

total 336792
drwxr-xr-x 2 root root      4096 Aug 24 14:49 .
drwxr-xr-x 5 root root      4096 Aug 24 14:48 ..
-rw-r--r-- 1 root root 344862509 Aug 24 14:49 102flowers.tgz
-rw-r--r-- 1 root root       502 Aug 24 14:49 imagelabels.mat


In [ ]:
import time, nbformat
from nbclient import NotebookClient

IZVOR = "main.ipynb"
IZLAZ = "main_executed.ipynb"

nb = nbformat.read(IZVOR, as_version=4)
kodnih = sum(1 for cell in nb.cells if cell.cell_type == "code")
print(f"celija ukupno {len(nb.cells)}, kodnih {kodnih}\n")

client = NotebookClient(nb, timeout=None, kernel_name="python3",
                        resources={"metadata": {"path": "."}})

t0 = time.time()
with client.setup_kernel():
    for i, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        t = time.time()
        try:
            client.execute_cell(cell, i)
        except Exception as e:
            nbformat.write(nb, IZLAZ)
            print(f"\nPALO NA CELIJI {i} posle {(time.time()-t0)/60:.1f} min")
            print(type(e).__name__, str(e)[:600])
            raise
        dt = time.time() - t
        if dt > 5:
            print(f"  celija {i:3}: {dt:6.0f} s   (ukupno {(time.time()-t0)/60:5.1f} min)", flush=True)
            nbformat.write(nb, IZLAZ)

nbformat.write(nb, IZLAZ)
print(f"\nGOTOVO, ceo notebook je prosao za {(time.time()-t0)/60:.1f} min")

celija ukupno 188, kodnih 122

  celija  39:     30 s   (ukupno   0.7 min)
  celija  42:      6 s   (ukupno   0.9 min)
  celija  45:    174 s   (ukupno   3.8 min)
  celija  54:   1304 s   (ukupno  25.6 min)
  celija  57:   1299 s   (ukupno  47.3 min)
  celija  59:      6 s   (ukupno  47.4 min)
  celija  71:    132 s   (ukupno  49.6 min)
  celija  74:    302 s   (ukupno  54.6 min)
  celija  79:    302 s   (ukupno  59.7 min)
  celija  84:    534 s   (ukupno  68.6 min)
  celija  91:    546 s   (ukupno  77.7 min)
  celija  99:  14783 s   (ukupno 324.2 min)
  celija 106:      8 s   (ukupno 324.4 min)
  celija 134:      7 s   (ukupno 324.6 min)
  celija 137:     13 s   (ukupno 324.8 min)
  celija 158:      8 s   (ukupno 325.0 min)
  celija 161:     13 s   (ukupno 325.2 min)
  celija 171:      8 s   (ukupno 325.4 min)
  celija 184:      8 s   (ukupno 325.6 min)

GOTOVO, ceo notebook je prosao za 325.6 min


In [ ]:
import re, nbformat
nb = nbformat.read("main_executed.ipynb", as_version=4)

trazeni = ["Test accuracy", "Best validation accuracy", "Validation accuracy",
           "Best hyperparameters", "Best combination", "Loaded existing",
           "Number of images", "Total images", "Train batches", "accuracy  "]
print("=== SVI KLJUCNI ISPISI IZ PUNOG PROLAZA ===\n")
for i, cell in enumerate(nb.cells):
    for o in cell.get("outputs", []):
        t = "".join(o.get("text", [])) if o.get("output_type") == "stream" else ""
        for l in t.splitlines():
            if any(k in l for k in trazeni):
                print(f"[{i:3}] {l.strip()[:110]}")

=== SVI KLJUCNI ISPISI IZ PUNOG PROLAZA ===

[  5] Total images: 8189
[ 14] Number of images after deleting corrupted:  8189
[ 41] Validation accuracy for classic KNN (k = 5):  0.14890154597233524
[ 42] Validation accuracy for RBF KNN (k = 15):  0.14727420667209112
[ 46] Best hyperparameters:  {'k': 5, 'gamma': 0.001, 'acc': 0.16273393002441008}
[ 48] Test accuracy:  0.16517493897477625
[ 48] accuracy                         0.1652      1229
[ 54] Validation accuracy for Chi2 KNN (k = 15): 0.14564686737184704
[ 60] Best hyperparameters:  {'k': 5, 'gamma': 0.0001, 'acc': 0.15215622457282343}
[ 76] Validation accuracy for combined HOG + HSV KNN (k = 15):  0.4019528071602929
[ 84] Best combination for HOG + HSV:  {'gamma_hog': np.float64(0.02242327263072366), 'gamma_hsv': np.float64(2.5138
[ 85] Test accuracy for combined HOG + HSV KNN:  0.46867371847030104
[ 91] Validation accuracy for RBF SVM: 0.3002441008950366
[ 98] Validation accuracy for combined HOG+HSV SVM: 0.54759967453214
[ 99] 

In [1]:
!zip -q -r rezultat_punog_prolaza.zip main_executed.ipynb assets/knn_rbf_grid.json assets/knn_chi2_grid.json assets/knn_combined_grid.json assets/svm_combined_grid.json assets/error_analysis_v2.npz assets/*.png
!ls -la rezultat_punog_prolaza.zip
from google.colab import files
files.download('rezultat_punog_prolaza.zip')


zip error: Nothing to do! (try: zip -q -r rezultat_punog_prolaza.zip . -i main_executed.ipynb assets/knn_rbf_grid.json assets/knn_chi2_grid.json assets/knn_combined_grid.json assets/svm_combined_grid.json assets/error_analysis_v2.npz assets/*.png)
ls: cannot access 'rezultat_punog_prolaza.zip': No such file or directory


FileNotFoundError: Cannot find file: rezultat_punog_prolaza.zip